[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_computing/02_error_propagation_and_stability_tricks/first_principles.ipynb)

# Topic 02: Error Propagation and Stability Tricks

## 1. First-Principles Intuition & Motivation

A single floating-point operation errs by at most one part in $10^{16}$ (binary64). Naively, even a billion operations should leave 7 good digits. Yet real programs routinely produce answers with *zero* correct digits. Two mechanisms are responsible:

1. **Amplification**: a subtraction of nearly equal quantities rescales tiny prior errors into the leading digits.
2. **Accumulation**: long chains of operations compound $(1+\delta)$ factors, and *biased* errors add coherently rather than cancelling.

The remedy is never "use more precision" as a first resort — it is to *restructure the computation*. This module builds the error calculus that predicts failure, then catalogs the classical restructurings that prevent it.

### The formula is not the algorithm

Consider $f(x) = \sqrt{x + 1} - \sqrt{x}$ at $x = 10^{12}$. Both square roots are $\approx 10^{6}$ and agree to 12 digits; their computed difference retains only ~4 significant digits. The algebraically identical form

$$
f(x) = \frac{1}{\sqrt{x+1} + \sqrt{x}}
$$

computes the same value to full 16-digit accuracy — the subtraction has been performed *symbolically*, where it is exact.

One mathematical function, two algorithms, twelve digits of difference. Numerical computing is the study of this gap.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Forward and backward error)

Let $y = f(x)$ be the exact result and $\hat{y}$ the computed result.

- **Forward error**: the (relative) distance in output space, $\frac{\lvert \hat{y} - y \rvert}{\lvert y \rvert}$.
- **Backward error**: the size of the smallest input perturbation explaining the output, $\eta(\hat{y}) = \min \left\{ \frac{\lvert \Delta x \rvert}{\lvert x \rvert} : \hat{y} = f(x + \Delta x) \right\}$.

An algorithm is **backward stable** if it always produces $\hat{y} = f(x + \Delta x)$ with $\frac{\lVert \Delta x \rVert}{\lVert x \rVert} = O(u)$: *the exact answer to a nearby question*.

### Theorem 2.2 (The $\gamma_n$ lemma — composing rounding errors)

If $\lvert \delta_i \rvert \le u$ for $i = 1, \dots, n$ and $nu \lt 1$, then

$$
\prod_{i=1}^{n} (1 + \delta_i)^{\pm 1} = 1 + \theta_n, \qquad \lvert \theta_n \rvert \le \gamma_n := \frac{nu}{1 - nu}
$$

This is the workhorse notation of Higham's error analysis: any chain of $n$ multiplications, divisions, and their roundings perturbs the result by a single factor bounded by $\gamma_n \approx nu$ for $nu \ll 1$.

### Theorem 2.3 (Recursive summation error bound)

Computing $S_n = \sum_{i=1}^{n} x_i$ by the loop `s += x[i]` yields a computed sum satisfying

$$
\hat{S}_n = \sum_{i=1}^{n} x_i (1 + \epsilon_i), \qquad \lvert \epsilon_i \rvert \le \gamma_{n-1}
$$

hence the forward error bound

$$
\lvert \hat{S}_n - S_n \rvert \le \gamma_{n-1} \sum_{i=1}^{n} \lvert x_i \rvert
$$

The ratio $\sum \lvert x_i \rvert / \lvert \sum x_i \rvert$ is the *condition number of summation*: large when massive cancellation occurs in the exact sum.

### Theorem 2.4 (Pairwise and compensated summation)

- **Pairwise (cascade) summation** — recursively summing halves — satisfies the same model with $\lvert \epsilon_i \rvert \le \gamma_{\lceil \log_2 n \rceil}$: error growth $O(u \log n)$ instead of $O(un)$. NumPy's `np.sum` uses a blocked pairwise scheme (block size 128).
- **Kahan compensated summation** maintains a running correction and satisfies

$$
\hat{S}_n = \sum_{i=1}^{n} x_i (1 + \mu_i), \qquad \lvert \mu_i \rvert \le 2u + O(nu^2)
$$

— error effectively *independent of $n$* until $n \sim 1/u$.

### Definition 2.5 (Error-free transformations)

For floats $a, b$, **Fast2Sum** (Dekker) computes floats $s, t$ with

$$
s = \mathrm{fl}(a + b), \qquad s + t = a + b \text{ exactly}
$$

using 3 flops (when $\lvert a \rvert \ge \lvert b \rvert$): `s = a + b; z = s - a; t = b - z`. The rounding error of an addition is itself a representable float, recoverable in exact arithmetic. Kahan summation is Fast2Sum applied inside a loop, feeding each step's $t$ back into the next addition.

### Theorem 2.6 (Cancellation error bound)

Let $x, y \gt 0$ carry prior relative errors: $\hat{x} = x(1 + \delta_x)$, $\hat{y} = y(1 + \delta_y)$ with $\lvert \delta_x \rvert, \lvert \delta_y \rvert \le \epsilon$. Then even if the subtraction itself is exact,

$$
\frac{\lvert (\hat{x} - \hat{y}) - (x - y) \rvert}{\lvert x - y \rvert} \le \epsilon \cdot \frac{x + y}{\lvert x - y \rvert}
$$

The amplification factor $\frac{x+y}{\lvert x-y \rvert}$ is huge precisely when $x \approx y$. This *is* the condition number of subtraction — cancellation is ill-conditioning of the subtraction map, not misbehavior of the hardware.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Derivation 3.1: The cancellation bound and the digit-loss rule

**Proof of Theorem 2.6.** The error of the difference is

$$
(\hat{x} - \hat{y}) - (x - y) = x\delta_x - y\delta_y
$$

so by the triangle inequality

$$
\lvert (\hat{x} - \hat{y}) - (x - y) \rvert \le \epsilon (x + y)
$$

Dividing by $\lvert x - y \rvert$ gives the claim. $\blacksquare$

**Digit-loss rule.** If $x$ and $y$ agree to $k$ significant digits, then $\lvert x - y \rvert \approx 10^{-k}(x+y)/2$, so the amplification factor is $\approx 2 \times 10^{k}$: *the result loses about $k$ digits relative to the accuracy of the inputs*. With binary64 inputs accurate to 16 digits, agreement to 12 digits leaves only ~4 trustworthy digits.

### Derivation 3.2: Proof of the $\gamma_n$ lemma

**Claim**: $\lvert \delta_i \rvert \le u$, $nu \lt 1$ $\implies$ $\prod_{i=1}^{n}(1+\delta_i) = 1 + \theta_n$ with $\lvert \theta_n \rvert \le \frac{nu}{1-nu}$.

**Upper side.** By AM–GM-type expansion,

$$
\prod_{i=1}^{n}(1 + \delta_i) \le (1 + u)^n = \sum_{k=0}^{n} \binom{n}{k} u^k \le \sum_{k=0}^{n} (nu)^k \cdot \frac{1}{n^k}\binom{n}{k} n^k \le \sum_{k \ge 0} (nu)^k = \frac{1}{1 - nu}
$$

using $\binom{n}{k} \le n^k$. Hence $\theta_n \le \frac{1}{1-nu} - 1 = \frac{nu}{1-nu}$.

**Lower side.** Similarly $\prod (1 + \delta_i) \ge (1-u)^n \ge 1 - nu \ge 1 - \frac{nu}{1-nu}$, using Bernoulli's inequality $(1-u)^n \ge 1 - nu$. Combining, $\lvert \theta_n \rvert \le \gamma_n$. The same bound absorbs factors $(1+\delta_i)^{-1}$ since $\frac{1}{1+\delta} = 1 + \delta'$ with $\lvert \delta' \rvert \le \frac{u}{1-u}$. $\blacksquare$

**Reading**: for $nu \ll 1$, $\gamma_n \approx nu$ — error budgets simply *add* along an operation chain, at first order.

### Derivation 3.3: Recursive summation — where the $x_1$ term pays most

Unrolling `s = ((x1 + x2) + x3) + ... + xn` with one rounding per addition:

$$
\hat{S}_n = \left( \cdots \left( (x_1 + x_2)(1+\delta_2) + x_3 \right)(1+\delta_3) \cdots + x_n \right)(1+\delta_n)
$$

Each $x_i$ (for $i \ge 2$) is multiplied by $\prod_{j=i}^{n} (1+\delta_j)$, and $x_1$ by $\prod_{j=2}^{n}(1+\delta_j)$. By the $\gamma$ lemma,

$$
\hat{S}_n = x_1(1 + \theta_{n-1}) + \sum_{i=2}^{n} x_i (1 + \theta_{n-i+1}), \qquad \lvert \theta_k \rvert \le \gamma_k
$$

**Two consequences.**

1. The worst coefficient $\gamma_{n-1}$ multiplies the *earliest* terms: summing in increasing order of magnitude puts the smallest numbers under the largest error factors — the classical "sort ascending before summing" heuristic.
2. The forward error bound $\lvert \hat{S}_n - S_n \rvert \le \gamma_{n-1} \sum \lvert x_i \rvert$ is scale-aware: with mixed signs and heavy cancellation, the *relative* error can be arbitrarily bad even though the absolute bound is modest.

### Derivation 3.4: Kahan summation — algorithm and error-bound sketch

**Algorithm** (running sum $s$, compensation $c$):

```text
s = 0; c = 0
for x in data:
    y = x - c        # apply stored correction
    t = s + y        # big + small: low bits of y are lost...
    c = (t - s) - y  # ...but recovered exactly here
    s = t
```

**Why $c$ recovers the lost bits.** For $\lvert s \rvert \ge \lvert y \rvert$, Fast2Sum guarantees that $t = \mathrm{fl}(s + y)$ and the sequence `(t - s) - y` computes *exactly* the rounding error $t - (s + y)$: both subtractions are exact by Sterbenz-type arguments. Thus $c$ holds precisely what the addition dropped, and subtracting $c$ from the next summand re-injects it.

**Error bound sketch.** Let $S_n = \sum x_i$. One shows by induction that the pair $(s, c)$ satisfies the invariant that $s - c$ equals the exact partial sum up to second-order terms: each iteration commits only the $O(u^2)$ error of representing the compensation itself, plus one $u$-level error on the *final* stored $s$. Summing the second-order terms over $n$ steps yields Knuth/Goldberg's bound

$$
\hat{S}_n = \sum_{i=1}^{n} x_i (1 + \mu_i), \qquad \lvert \mu_i \rvert \le 2u + O(nu^2)
$$

Compared with $\gamma_{n-1} \approx nu$ for the naive loop, the linear-in-$n$ term has been demoted to second order: for binary64 and $n \lt 10^{10}$, Kahan summation is effectively exact to one ulp of the condition of the data. $\blacksquare$ (Full proof: Higham 2002, Thm. 4.8.)

### Derivation 3.5: The stable quadratic formula

For $ax^2 + bx + c = 0$ with $b^2 \gg 4ac$, the textbook roots

$$
x_{1,2} = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}
$$

suffer cancellation in whichever numerator subtracts nearly equal quantities: $\sqrt{b^2 - 4ac} \approx \lvert b \rvert$, so one root computes as a tiny difference of two $O(\lvert b \rvert)$ values.

**Fix.** Compute the *large*-magnitude root with the sign that adds:

$$
q = -\tfrac{1}{2}\left( b + \operatorname{sign}(b)\sqrt{b^2 - 4ac} \right), \qquad x_1 = \frac{q}{a}
$$

then recover the small root from Vieta's exact product $x_1 x_2 = c/a$:

$$
x_2 = \frac{c}{q}
$$

**Error accounting.** $x_1$ involves only additions of same-signed quantities and a square root — every step well-conditioned, so $x_1$ is accurate to $O(u)$. Then $x_2 = c/q$ is a single division of accurate quantities: also $O(u)$. The cancellation has been replaced by an *exact algebraic identity*.

**Numerical example** ($a = 1$, $b = 10^{8}$, $c = 1$): the naive small root returns $\mathrm{fl}(-10^{8} + \sqrt{10^{16} - 4})/2 \approx -7.45 \times 10^{-9}$ (wrong by 25%), while $c/q$ gives $-1.0000000000000000 \times 10^{-8}$ — correct to all digits.

### Derivation 3.6: Why `log1p` and `expm1` exist

For small $\lvert x \rvert$, the Taylor series give

$$
\log(1 + x) = x - \frac{x^2}{2} + \frac{x^3}{3} - \cdots, \qquad e^{x} - 1 = x + \frac{x^2}{2} + \frac{x^3}{6} + \cdots
$$

Both functions are $\approx x$: the *deviation* is the signal. But the naive evaluations first form $\mathrm{fl}(1 + x)$ or $\mathrm{fl}(e^x)$, which round to a neighborhood of 1 with absolute grid spacing $\varepsilon_{\mathrm{mach}}$. For $x = 10^{-12}$ (binary64):

$$
\mathrm{fl}(1 + x) = 1 + \tilde{x}, \qquad \lvert \tilde{x} - x \rvert \le \tfrac{1}{2}\varepsilon_{\mathrm{mach}} \approx 1.1 \times 10^{-16}
$$

so $\tilde{x}$ carries a relative error up to $\approx 10^{-4}$ — twelve digits lost before the logarithm is even called. The library functions `log1p(x)` and `expm1(x)` take $x$ *directly* and evaluate the series (or an equivalent argument-reduced kernel), returning full relative accuracy $O(u)$ for all $x$.

**Rule**: whenever an API hands you a quantity near a reference point (probability near 1, return near 0, ratio near 1), keep and process the *offset*, never the absolute value.

### Derivation 3.7: One-pass variance disasters and Welford's update

The textbook identity $\mathrm{Var}(x) = \overline{x^2} - \bar{x}^2$ subtracts two $O(\bar{x}^2)$ quantities. If mean $\mu = 10^{6}$ and standard deviation $\sigma = 1$, the two terms agree to 12 digits — in binary32 arithmetic the result is pure noise and can even be *negative*.

**Welford's online algorithm** maintains the running mean $m_k$ and centered sum of squares $M_k$:

$$
m_k = m_{k-1} + \frac{x_k - m_{k-1}}{k}, \qquad M_k = M_{k-1} + (x_k - m_{k-1})(x_k - m_k)
$$

with $\mathrm{Var} = M_n / n$ (or $M_n/(n-1)$). Every subtraction is a *residual* $x_k - m$: small minus small, no cancellation of large quantities. One can verify the recurrence reproduces $M_n = \sum (x_i - m_n)^2$ exactly in real arithmetic by induction, while its floating-point error stays $O(u)$ relative to $M_n$ itself rather than to $n\mu^2$.

This exact pattern — track residuals, not absolutes — reappears in BatchNorm running statistics, Adam's second-moment estimates, and streaming feature normalization.

## 4. Computational & Algorithmic Insights

### 4.1 What NumPy and friends actually do

- `np.sum` uses **blocked pairwise summation** for float arrays (blocks of 128, recursion above): error $O(u \log n)$ at essentially the speed of the naive loop, since blocking preserves vectorization.
- `math.fsum` (Python) implements Shewchuk's exact summation with an expansion of partial sums — correctly rounded result, several times slower.
- `np.hypot(a, b)` computes $\sqrt{a^2 + b^2}$ without overflow by factoring out $\max(\lvert a \rvert, \lvert b \rvert)$: for $a = 10^{200}$, the naive square overflows while `hypot` returns the right answer.
- `np.log1p`, `np.expm1`, `scipy.special.logsumexp`, `scipy.special.xlogy` are the vectorized stable kernels; reaching for them is usually the entire fix.
- Means: `np.mean` divides once at the end; a running `s += x; s /= n` style loop commits $n$ extra roundings.

### 4.2 Choosing a summation strategy

| Strategy | Error growth | Cost | When to use |
|---|---|---|---|
| Recursive loop | $O(nu)$ | 1 add/term | Small $n$, or accumulator precision ≫ data precision |
| Sorted ascending | $O(nu)$, smaller constant | sort + adds | Mixed magnitudes, one-shot sums |
| Pairwise (`np.sum`) | $O(u \log n)$ | 1 add/term | Default — free accuracy |
| Kahan / compensated | $2u + O(nu^2)$ | 4 flops/term | Long accumulations, ill-conditioned sums |
| `math.fsum` / Shewchuk | exact (correctly rounded) | ~5–10× | Testing, reference values, geometry predicates |
| Wider accumulator (fp32 data, fp64 acc) | $O(nu_{\text{wide}})$ | ~free on CPU | Deep learning reductions, BatchNorm statistics |

The last row is the deep-learning workhorse: summing fp16/bf16 values into an fp32 accumulator multiplies the effective $n$ headroom by $u_{16}/u_{32} \approx 2^{13}$.

### 4.3 Testing numerical code

- **Golden values**: compare against `math.fsum`, `mpmath` (arbitrary precision), or closed forms.
- **Perturbation testing**: run in float32 and float64; if results differ far beyond $u_{32}/u_{64}$ scaling, the algorithm is unstable.
- **Backward-error checks**: for a linear solve, verify the residual $\lVert b - A\hat{x} \rVert / (\lVert A \rVert \lVert \hat{x} \rVert) = O(u)$ — achievable even when the forward error is large (Topic 03).
- **Property tests**: variance is nonnegative; probabilities sum to 1 within $nu$; energy drift in symplectic integrators is bounded.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Loss averaging over large batches and long epochs

A training loop that accumulates `total_loss += batch_loss` in fp16 stalls once `total_loss` grows so large that $\mathrm{ulp}(\text{total}) \gt \text{batch loss}$: with fp16's 11-bit significand, after roughly $2^{11}$ comparable-magnitude additions, subsequent terms round entirely away — the reported epoch loss silently freezes. The standard fixes are exactly this module's toolbox: accumulate in fp32 (wider accumulator), or average hierarchically (pairwise), or keep per-batch losses and reduce once.

### 5.2 Softmax cross-entropy via `log1p`-style rewrites

The loss $-\log p_y$ with $p_y = \mathrm{softmax}(z)_y$ naively computes $\log$ of an exponential ratio — overflow, underflow, and cancellation stacked together. The stable form fuses the algebra symbolically:

$$
-\log p_y = -z_y + \log \sum_{j} e^{z_j} = -z_y + m + \log \sum_{j} e^{z_j - m}, \qquad m = \max_j z_j
$$

Every framework's `cross_entropy_with_logits` is this identity; Topic 05 proves its correctness and error bound. It is the `log1p` philosophy at scale: never exponentiate and then take logs — restructure so logs and exps annihilate on paper.

### 5.3 Physics: energy drift and compensated integrators

In long-horizon orbital mechanics and molecular dynamics, the summation error of position updates acts like a spurious force. Symplectic integrators bound the *systematic* energy error, but floating-point bias remains; production N-body codes (e.g. REBOUND's IAS15) use compensated (Kahan) accumulation of the state vector so that round-off grows like $\sqrt{n}$ (random walk) instead of $n$ (bias) over $10^{9}$ steps — the difference between planetary ephemerides that hold for a gigayear and ones that drift visibly.

### 5.4 Finance: present values and `expm1`

Continuous compounding at tiny rates, $V = P(e^{r\tau} - 1)$ with $r\tau \sim 10^{-9}$ (overnight rates), loses all digits under naive evaluation but is exact-to-$u$ with `expm1`. The same applies to log-returns $\log(P_1/P_0) = \mathrm{log1p}((P_1 - P_0)/P_0)$ for small moves — the version quant libraries mandate, because portfolio aggregation sums millions of such terms and biased per-term error compounds linearly (Sec. 3.3).

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Location |
|---|---|---|
| $\gamma_n$ notation and composition lemma | Higham, *Accuracy and Stability of Numerical Algorithms* (2002) | Lemma 3.1, Sec. 3.1–3.4 |
| Summation error bounds (recursive, pairwise, Kahan) | Higham (2002) | Ch. 4, Thm. 4.8 |
| Cancellation and guard digits | Goldberg (1991), *What Every Computer Scientist Should Know About Floating-Point Arithmetic* | Sec. "Cancellation" |
| Compensated summation, original note | Kahan, CACM 8(1) (1965) | p. 40 |
| Fast2Sum / TwoSum error-free transforms | Muller et al., *Handbook of Floating-Point Arithmetic* (2018) | Ch. 4 |
| Backward stability framework | Trefethen & Bau, *Numerical Linear Algebra* (1997) | Lectures 14–15 |
| Online variance | Welford, Technometrics 4(3) (1962); Chan, Golub & LeVeque (1983) | — |
| Exact summation expansions | Shewchuk, *Adaptive Precision Floating-Point Arithmetic* (1997) | Sec. 2 |
| Mixed-precision accumulation in DL | Micikevicius et al., *Mixed Precision Training* (ICLR 2018) | Sec. 3.3 |

**Continue to** [Topic 03: Conditioning and Condition Numbers](../03_conditioning_and_condition_numbers/README.md) — the input-sensitivity half of the accuracy story, and the bridge from backward to forward error.